# 03 — Isolation Metrics (Tier A)  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Set PROJECT_ID → **Runtime → Run all**. All queries read your own `frame_events_*` / `repo_month_events` tables — cheap (well under the free tier), no more GH Archive scans.

Computes four socio-technical **isolation signals** at repo-month granularity, assembles a metrics panel, builds the ecosystem monthly time series, and plots a first look at the trends 2018–2026.

| Metric | Meaning | Operationalization |
|---|---|---|
| **CD** — Collaboration Density | interaction richness per unit of code | (human issue+review comments + reviews) / (commits + 1) |
| **SWR** — Solitary Workflow Ratio | share of PRs merged with no human dialogue | fraction of PRs with zero non-author review/comment |
| **CR** — Communication Reciprocity | do interactions go both ways | directed-graph reciprocity of the interaction network |
| **HIE** — Human Interaction Entropy | is dialogue spread or centralized | normalized Shannon entropy of contributors' interaction shares |

All bot activity (`is_bot`) is excluded from the interaction signals so we measure *human* collaboration.


## Config — edit PROJECT_ID (same id as before)

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"
# ==========================================================================
RELIABLE_END = "202506"          # trim the unreliable data-boundary tail (commits collapse after ~mid-2025)
KNOWN_GAP_MONTHS = ["202110"]    # GH Archive outage month — excluded from trend series
BQ_DATASET = "seis"; BQ_LOCATION = "US"; MAX_SCAN_GB = 500
DRIVE_BASE = "/content/drive/MyDrive/ICSE_SEIS"
assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded. PROJECT_ID =", PROJECT_ID)


## Setup — auth, BigQuery, Drive, helpers
Sign-in popup — choose **ANON@institution**.

In [ ]:
import os, json, hashlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
for sub in ["", "/data/derived", "/figures", "/tables", "/checkpoints"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
CKPT_DIR = f"{DRIVE_BASE}/checkpoints"

def _atomic(path, text):
    tmp=path+".tmp"; open(tmp,"w").write(text); os.replace(tmp,path)
def load_json(n,d):
    p=f"{CKPT_DIR}/{n}"; return json.load(open(p)) if os.path.exists(p) else d
def save_json(n,o): _atomic(f"{CKPT_DIR}/{n}", json.dumps(o,indent=2,default=str))
def manifest_mark(name,key):
    s=set(load_json(f"manifest_{name}.json",[])); s.add(str(key)); save_json(f"manifest_{name}.json",sorted(s))
_jr = load_json("bq_jobs.json", {})
def _sjr(): save_json("bq_jobs.json", _jr)

def run_query(sql, destination=None, write="WRITE_TRUNCATE", to_dataframe=False, label=None, job_key=None, cluster=None):
    tag=f"[{label}] " if label else ""
    sqlh=hashlib.md5(sql.encode()).hexdigest()   # reattach only if the QUERY is unchanged
    if job_key and job_key in _jr and _jr[job_key].get("sql_hash")==sqlh:
        try:
            prev=client.get_job(_jr[job_key]["job_id"], location=BQ_LOCATION)
            if prev.state=="DONE" and prev.error_result is None:
                print(f"{tag}reusing job {_jr[job_key]['job_id']} (identical query)")
                r=prev.result(); return r.to_dataframe() if to_dataframe else r
        except Exception as e: print(f"{tag}(reattach failed: {e})")
    gb=client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True)).total_bytes_processed/1e9
    print(f"{tag}scan ~{gb:,.1f} GB (~${gb/1000*6.25:,.2f})")
    cfg=bigquery.QueryJobConfig(maximum_bytes_billed=int(max(MAX_SCAN_GB,gb*1.2)*1e9))
    if destination: cfg.destination=f"{PROJECT_ID}.{BQ_DATASET}.{destination}"; cfg.write_disposition=write
    if cluster: cfg.clustering_fields=cluster
    job=client.query(sql, job_config=cfg)
    if job_key: _jr[job_key]={"job_id":job.job_id,"sql_hash":sqlh}; _sjr()
    r=job.result(); print(f"{tag}done -> {destination or 'df'}")
    return r.to_dataframe() if to_dataframe else r

P  = f"{PROJECT_ID}.{BQ_DATASET}"
FE = f"`{PROJECT_ID}.{BQ_DATASET}.frame_events_*`"
# PullRequestReviewEvent is EXCLUDED: it has no GH Archive coverage before ~2020-08,
# which would create a spurious mid-2020 break. We keep review COMMENTS (the substantive
# inline discussion) plus issue and commit comments — all stably covered 2018-2025.
COMMENT_TYPES = "('PullRequestReviewCommentEvent','IssueCommentEvent','CommitCommentEvent')"
print("Ready. Dataset:", P, "| interaction types:", COMMENT_TYPES)


## Metric — Solitary Workflow Ratio (SWR)
A PR is *solitary* if it received **no interaction from anyone other than its author** (no review, no review comment, no non-author issue comment). SWR = solitary PRs / all PRs opened that month. Higher SWR = more coding alone.

**Bots excluded from PR openers** (`NOT is_bot`): dependabot/renovate open huge waves of auto-merged dependency PRs that are 'solitary' by construction — including them would inflate SWR with automation rather than human isolation.

In [ ]:
swr_sql = f"""
WITH prs AS (
  SELECT repo_id, number, MIN(created_at) AS opened_at
  FROM {FE}
  WHERE type='PullRequestEvent' AND action='opened' AND number IS NOT NULL
        AND NOT is_bot
  GROUP BY repo_id, number),
inter AS (
  SELECT repo_id, number,
         COUNTIF(target_author IS NULL OR actor_login != target_author) AS ext
  FROM {FE}
  WHERE type IN {COMMENT_TYPES} AND number IS NOT NULL AND NOT is_bot
  GROUP BY repo_id, number)
SELECT p.repo_id, FORMAT_TIMESTAMP('%Y%m', p.opened_at) AS ym,
       COUNT(*) AS prs,
       COUNTIF(IFNULL(i.ext,0)=0) AS solitary_prs,
       SAFE_DIVIDE(COUNTIF(IFNULL(i.ext,0)=0), COUNT(*)) AS swr
FROM prs p LEFT JOIN inter i USING(repo_id, number)
GROUP BY repo_id, ym
"""
run_query(swr_sql, destination="metric_swr", label="SWR", job_key="metric_swr", cluster=["repo_id"])


## Metric — Communication Reciprocity (CR)
Build the directed interaction graph per repo-month (an edge A→B means A commented on / reviewed B's thread). CR = fraction of edges A→B for which B→A also exists. Low CR = one-directional, broadcast-style interaction.

In [ ]:
cr_sql = f"""
WITH edges AS (
  SELECT DISTINCT repo_id, FORMAT_TIMESTAMP('%Y%m', created_at) AS ym,
         actor_login AS src, target_author AS dst
  FROM {FE}
  WHERE type IN {COMMENT_TYPES} AND NOT is_bot
        AND actor_login IS NOT NULL AND target_author IS NOT NULL
        AND actor_login != target_author)
SELECT e.repo_id, e.ym,
       COUNT(*) AS dir_edges,
       COUNTIF(r.src IS NOT NULL) AS mutual_edges,
       SAFE_DIVIDE(COUNTIF(r.src IS NOT NULL), COUNT(*)) AS reciprocity
FROM edges e
LEFT JOIN edges r ON e.repo_id=r.repo_id AND e.ym=r.ym AND e.src=r.dst AND e.dst=r.src
GROUP BY e.repo_id, e.ym
"""
run_query(cr_sql, destination="metric_cr", label="CR", job_key="metric_cr", cluster=["repo_id"])


## Metric — Human Interaction Entropy (HIE)
Per repo-month, the normalized Shannon entropy of how interaction events are distributed across contributors. HIE≈1 = many voices share the dialogue; HIE→0 = one or two voices dominate (centralized / isolated). Single-voice months (n=1) are set to 0.

In [ ]:
hie_sql = f"""
WITH acts AS (
  SELECT repo_id, FORMAT_TIMESTAMP('%Y%m', created_at) AS ym, actor_login, COUNT(*) AS cbt
  FROM {FE}
  WHERE type IN {COMMENT_TYPES} AND NOT is_bot AND actor_login IS NOT NULL
  GROUP BY repo_id, ym, actor_login),
w AS (
  SELECT repo_id, ym, cbt,
         SUM(cbt) OVER (PARTITION BY repo_id, ym) AS tot,
         COUNT(*) OVER (PARTITION BY repo_id, ym) AS n
  FROM acts)
SELECT repo_id, ym, ANY_VALUE(n) AS n_interactors,
       -SUM((cbt/tot)*LN(cbt/tot)) AS hie_entropy,
       IFNULL(SAFE_DIVIDE(-SUM((cbt/tot)*LN(cbt/tot)), NULLIF(LN(ANY_VALUE(n)),0)), 0) AS hie_norm
FROM w GROUP BY repo_id, ym
"""
run_query(hie_sql, destination="metric_hie", label="HIE", job_key="metric_hie", cluster=["repo_id"])


## Assemble the metrics panel
Join the three metric tables onto `repo_month_events` (which supplies commits/comment counts for **CD** and the control variables), plus repo-level covariates from the sampling frame. `metrics_panel` is the repo-month analysis table for the statistical models.

In [ ]:
panel_sql = f"""
SELECT b.repo_id, b.ym,
  b.commits, b.pushes, b.pr_opened,
  b.issue_comments_human, b.review_comments_human, b.reviews, b.distinct_actors_human,
  SAFE_DIVIDE(b.issue_comments_human + b.review_comments_human, b.commits + 1) AS cd,
  (b.commits>0 OR b.pr_opened>0 OR b.issue_comments_human>0
   OR b.review_comments_human>0) AS active,
  sw.prs, sw.solitary_prs, sw.swr,
  cr.dir_edges, cr.reciprocity,
  hi.n_interactors, hi.hie_norm,
  f.total_stars, f.age_days, f.peak_year_actors, f.primary_language
FROM `{P}.repo_month_events` b
LEFT JOIN `{P}.metric_swr` sw USING(repo_id, ym)
LEFT JOIN `{P}.metric_cr`  cr USING(repo_id, ym)
LEFT JOIN `{P}.metric_hie` hi USING(repo_id, ym)
LEFT JOIN `{P}.sampling_frame` f ON b.repo_id = f.repo_id
"""
run_query(panel_sql, destination="metrics_panel", label="panel", job_key="metrics_panel", cluster=["repo_id"])
print("metrics_panel built.")


## Ecosystem monthly time series
Aggregate each metric across all repos per month — this is the series the interrupted-time-series / AI-era analysis (notebook 09/10) will model, and what we plot below.

In [ ]:
ts_sql = f"""
SELECT ym,
  COUNT(*) AS repo_months,
  AVG(IF(active, cd, NULL)) AS cd_mean,      -- CD over development-active months only
  AVG(swr) AS swr_mean,                      -- SWR already defined only on PR months
  AVG(reciprocity) AS recip_mean,            -- CR/HIE already defined only on interaction months
  AVG(hie_norm) AS hie_mean
FROM `{P}.metrics_panel`
WHERE ym BETWEEN '201801' AND '{RELIABLE_END}'
  AND ym NOT IN UNNEST({KNOWN_GAP_MONTHS})
GROUP BY ym ORDER BY ym
"""
ts = run_query(ts_sql, to_dataframe=True, label="timeseries", job_key="timeseries")
ts["date"] = pd.to_datetime(ts["ym"] + "01", format="%Y%m%d")
ts.to_csv(f"{DRIVE_BASE}/tables/metrics_timeseries.csv", index=False)
print("Saved metrics_timeseries.csv"); ts.tail()


## First look — the four trends (2018–2026)
A quick, unstyled sanity plot (final publication figures come in notebook 11). The dashed line marks the ChatGPT release (Nov 2022) — the candidate AI-era break.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
series = [("cd_mean","Collaboration Density (CD)"),
          ("swr_mean","Solitary Workflow Ratio (SWR)"),
          ("recip_mean","Communication Reciprocity (CR)"),
          ("hie_mean","Human Interaction Entropy (HIE)")]
ai = pd.Timestamp("2022-11-01")
for a,(col,title) in zip(ax.flat, series):
    a.plot(ts["date"], ts[col])
    a.axvline(ai, ls="--", color="crimson", lw=1)
    a.set_title(title); a.grid(alpha=.3)
fig.suptitle("Socio-technical isolation signals across ~50k GitHub repos, 2018–2026", y=1.02)
fig.tight_layout()
fig.savefig(f"{DRIVE_BASE}/figures/first_look_trends.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figures/first_look_trends.png")


## Done ✅
Built: `metric_swr`, `metric_cr`, `metric_hie`, `metrics_panel` (repo-month analysis table), `metrics_timeseries` (+ CSV + first-look figure on Drive).

**Read the plot as a first signal, not a result** — the statistics that make it defensible (interrupted-time-series with placebo/negative-control checks, mixed-effects models, and the difference-in-differences AI-era identification) come in notebooks 09–10, after the network metrics (05) and the manual validation (07–08).

Next: **`04_deep_fetch.ipynb`** — GitHub-API deep fetch for the Tier-B subsample (comment text for the Mentorship Decay Index, exact contributor counts, and the primary-language enrichment that fixes the 94%-unknown gap).